# 方案三：BERT + CRF 多任务序列标注复现

把 ACOS 抽取拆成"逐字 BIO 标注 + 句子分类"：

```
bert-base-chinese（逐字向量）
   ├── tag_proj → CRF：5 标签 BIO，联合抽 Aspect / Opinion 片段
   ├── cat_head（MLP）：13 类 Category
   └── pol_head（MLP）：3 类 Polarity
损失 L = CRF损失 + 0.5×类别CE + 0.5×极性CE
```

| 指标 | dev（可本地复算） | 测试集（排行榜） |
| --- | --- | --- |
| 四元组精确匹配 F1 | **0.5918**（8 epoch） | **0.6198** |

> 训练使用 GPU；脚本产物路径全部写入 `src/config.py` 规定的位置（项目根目录 `data/bert/`、`outputs/`、`config/` 下）。权重属大文件，不随仓库提供，本 Notebook 会重新训练生成。

## Step 0. 环境与依赖

**国内机器先配清华镜像**可加速 5~10 倍：

```bash
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip config set global.trusted-host pypi.tuna.tsinghua.edu.cn
```

然后安装（本 Notebook 使用 GPU，先确认 torch 版本与 CUDA 匹配）：

```bash
# GPU 版 torch（按你的 CUDA 版本选 cu118 / cu121 / cu124）
pip install torch --index-url https://download.pytorch.org/whl/cu121

# 其余依赖走 requirements.txt 即可
pip install -r requirements.txt

# HuggingFace 直连慢时
export HF_ENDPOINT=https://hf-mirror.com
```

首次运行会自动下载 `bert-base-chinese`（约 400MB）。

In [ ]:
import importlib.util, os

need = {"torch": "torch", "transformers": "transformers", "pytorch-crf": "torchcrf"}
missing = [pkg for pkg, mod in need.items() if importlib.util.find_spec(mod) is None]
print("依赖检查:", "全部就绪" if not missing else f"缺少 {missing}")

# HuggingFace 直连较慢时取消注释：
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [ ]:
# 统一入口：定位项目根目录（notebook/ 的上一级），把 src/ 加入 import 路径
# 脚本产物路径保持原样（config.py 中的 BASE_DIR/outputs, BASE_DIR/data/...）
import os, sys, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 脚本产物写入的根目录（与 src/config.py 保持一致）
ARTIFACT_DIR = PROJECT_ROOT

# 默认使用 0 号 GPU；多卡机器可改成 "1" 等
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

print("项目根目录:", PROJECT_ROOT)
print("产物根目录:", ARTIFACT_DIR)

def run(cmd):
    """用当前 Python 解释器执行命令，工作目录为 PROJECT_ROOT。"""
    print(">>", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, env=os.environ.copy())

In [ ]:
# GPU 环境检查：本实验要求使用 CUDA
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

assert torch.cuda.is_available(), "未检测到 CUDA GPU：请在有 NVIDIA GPU 的实例上运行本 Notebook"
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("显存: %.1f GB" % (props.total_memory / 1e9))
print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
print("bf16 支持:", torch.cuda.is_bf16_supported())
torch.backends.cudnn.benchmark = True

## Step 1. 准备预处理数据

BERT 模型读取 `outputs/processed/Train_reviews_processed.csv` 与 `Train_labels_parsed.csv`。若尚未生成，本 cell 会自动先运行方案一的预处理脚本。

In [ ]:
import pandas as pd

need_files = [
    ARTIFACT_DIR / "outputs/processed/Train_reviews_processed.csv",
    ARTIFACT_DIR / "outputs/processed/Train_labels_parsed.csv",
]
if not all(p.exists() for p in need_files):
    print("缺少预处理产物，先运行 src/preprocess.py ...")
    run([sys.executable, "src/preprocess.py"])
else:
    print("预处理产物已存在")

train_proc = pd.read_csv(need_files[0])
labels = pd.read_csv(need_files[1])
print("训练评论:", train_proc.shape, "| 标签:", labels.shape)

## Step 2. 下载/确认基座 tokenizer

首次运行会从 HuggingFace 下载 `bert-base-chinese`（约 400MB）。先单独加载 tokenizer，便于尽早发现网络问题。

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("bert-base-chinese")
print("tokenizer 词表大小:", len(tok))
print("示例编码:", tok("物流很快", add_special_tokens=False).tokens())

## Step 3. 训练 BERT+CRF

默认超参（`config/model_bert.yaml` 缺失时使用代码默认值）：8 epoch、batch 8、max_len 256、BERT 主干 lr 2e-5、任务头 lr 1e-3、warmup 0.1、梯度裁剪 1.0、dev_ratio 0.15、seed 42。

```bash
python src/model_bert/model_bert.py
```

In [ ]:
# 完整训练（使用 GPU）
run([sys.executable, "src/model_bert/model_bert.py"])

# 快速冒烟测试（仅验证代码能跑通，不作为最终结果）：
# run([sys.executable, "src/model_bert/model_bert.py", "--epochs", "1", "--max-len", "128"])

## Step 4. 查看训练日志与 dev 指标

In [ ]:
text = (ARTIFACT_DIR / "outputs/logs/bert_report.txt").read_text(encoding="utf-8")
print(text)

In [ ]:
import json
snapshot = ARTIFACT_DIR / "config/model_bert_trained.yaml"
if snapshot.exists():
    print(snapshot.read_text(encoding="utf-8"))
vocab = json.loads((ARTIFACT_DIR / "data/bert/label_vocab.json").read_text(encoding="utf-8"))
print("label_vocab 键:", list(vocab.keys()))
print("BIO 标签:", vocab.get("bio_tags"))
print("权重文件存在:", (ARTIFACT_DIR / "data/bert/bert_crf.pt").exists())

## Step 5. 对测试集推理并校验提交文件

```bash
python src/model_bert/predict_bert.py
```

推理会加载 `data/bert/bert_crf.pt` 与 `label_vocab.json`，输出 `data/bert/Result.csv`（约 3452 个四元组）。

In [ ]:
run([sys.executable, "src/model_bert/predict_bert.py"])

In [ ]:
import pandas as pd
from pathlib import Path

COLS = ["id", "AspectTerms", "OpinionTerms", "Categories", "Polarities"]
CATS = {"整体", "使用体验", "功效", "价格", "物流", "气味", "包装",
        "真伪", "服务", "其他", "成分", "尺寸", "新鲜度"}
POLS = {"正面", "负面", "中性"}

def validate_result(path):
    p = Path(path)
    assert p.exists(), f"结果文件不存在: {p}"
    df = pd.read_csv(p, header=None, names=COLS, dtype=str)
    df["id"] = df["id"].str.strip().astype(int)
    test_ids = set(pd.read_csv("data/raw/TEST/Test_reviews.csv")["id"].astype(int))
    assert set(df["id"]) == test_ids, "测试集 id 覆盖不完整"
    assert set(df["Categories"].str.strip()) <= CATS, "出现非法类别"
    assert set(df["Polarities"].str.strip()) <= POLS, "出现非法极性"
    for c in COLS[1:]:
        df[c] = df[c].str.strip()
    print(f"[通过] {p}")
    print(f"  行数（四元组数）: {len(df)} | 覆盖 id: {df['id'].nunique()} | 密度: {len(df)/len(test_ids):.2f} 个/条")
    print("  类别分布:", dict(df["Categories"].value_counts()))
    print("  极性分布:", dict(df["Polarities"].value_counts()))
    return df

df = validate_result(ARTIFACT_DIR / "data/bert/Result.csv")
display(df.head(10))

## 结果解读与局限

- dev F1 从第 1 个 epoch 的 0.5331 稳步升到第 8 个 epoch 的 **0.5918**，测试集 **0.6198**，明显超过规则基线；
- 但 71% 的隐式方面无法用 BIO 标签表达，单网络“先标词再拼四元组”的结构在配对和隐式场景上存在天花板；
- 这正是后续 LLM 生成式方案与两阶段管道方案要解决的问题。